# Аналитика логов сайта

Ноутбук считывает файл `logs/debug.log`, разбирает JSON-логи приложения и рассчитывает минимальные метрики работы сайта.

## Что анализируем

- количество событий в логах;
- количество посещений сайта;
- максимальный номер посетителя;
- распределение решений приложения;
- частоту посещений по времени;
- наличие ошибок Redis и healthcheck.

## 1. Импорт библиотек

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 120)

## 2. Загрузка файла логов

Файл должен находиться в текущем проекте по пути:

```text
logs/debug.log
```

Если ноутбук лежит в корне проекта, менять путь не нужно.

In [ ]:
log_file = Path('logs/debug.log')

if not log_file.exists():
    raise FileNotFoundError(
        'Файл logs/debug.log не найден. Проверьте, что приложение запускалось и volume для логов настроен правильно.'
    )

records = []

with log_file.open('r', encoding='utf-8') as file:
    for line_number, line in enumerate(file, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            print(f'Строка {line_number} пропущена: некорректный JSON')

df = pd.DataFrame(records)
df.head()

## 3. Подготовка данных

Поле `data` содержит вложенные данные. Например, номер посетителя хранится как `data.count`, а выбранный сценарий — как `data.decision`.

In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

df['count'] = df['data'].apply(lambda x: x.get('count') if isinstance(x, dict) else None)
df['decision'] = df['data'].apply(lambda x: x.get('decision') if isinstance(x, dict) else None)
df['error'] = df['data'].apply(lambda x: x.get('error') if isinstance(x, dict) else None)

df.head(10)

## 4. Основные метрики сайта

Посещение сайта фиксируется событием `Counter incremented`.  
Событие `Decision selected` показывает, какой сценарий был выбран для посетителя.

In [ ]:
visits = df[df['message'] == 'Counter incremented'].copy()
decisions = df[df['message'] == 'Decision selected'].copy()
errors = df[df['error'].notna()].copy()

metrics = {
    'Всего строк в логах': len(df),
    'Количество посещений сайта': len(visits),
    'Максимальный номер посетителя': int(visits['count'].max()) if not visits.empty else 0,
    'Количество решений приложения': len(decisions),
    'Количество ошибок': len(errors),
    'Время первого события': df['datetime'].min(),
    'Время последнего события': df['datetime'].max(),
}

metrics_df = pd.DataFrame(metrics.items(), columns=['Метрика', 'Значение'])
metrics_df

### Пояснение к метрикам

- **Количество посещений сайта** — сколько раз пользователь открыл или обновил главную страницу.
- **Максимальный номер посетителя** — текущее значение счетчика Redis.
- **Количество решений приложения** — сколько раз бизнес-логика выбрала сценарий ответа.
- **Количество ошибок** — наличие проблем Redis или healthcheck.

## 5. Распределение решений приложения

Эта таблица показывает, какие сценарии чаще всего срабатывали: обычный посетитель, чётный посетитель, счастливый посетитель и т.д.

In [ ]:
decision_stats = (
    decisions['decision']
    .value_counts()
    .rename_axis('Решение')
    .reset_index(name='Количество')
)

if not decision_stats.empty:
    decision_stats['Доля, %'] = (decision_stats['Количество'] / decision_stats['Количество'].sum() * 100).round(2)

decision_stats

In [ ]:
if not decision_stats.empty:
    plt.figure(figsize=(10, 5))
    plt.bar(decision_stats['Решение'], decision_stats['Количество'])
    plt.title('Распределение решений приложения')
    plt.xlabel('Тип решения')
    plt.ylabel('Количество')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Нет данных о решениях приложения.')

## 6. Динамика посещений по времени

График показывает, как быстро пользователи обращались к сайту. Для лабораторной работы это обычно обновления страницы вручную.

In [ ]:
if not visits.empty:
    visits_by_second = visits.set_index('datetime').resample('1s').size().reset_index(name='Количество посещений')
    visits_by_second.head()
else:
    visits_by_second = pd.DataFrame(columns=['datetime', 'Количество посещений'])
    print('Посещения не найдены.')

In [ ]:
if not visits_by_second.empty:
    plt.figure(figsize=(10, 5))
    plt.plot(visits_by_second['datetime'], visits_by_second['Количество посещений'], marker='o')
    plt.title('Посещения сайта по секундам')
    plt.xlabel('Время')
    plt.ylabel('Количество посещений')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Нет данных для построения графика.')

## 7. Проверка ошибок

Если ошибок нет, значит приложение успешно подключалось к Redis и корректно обрабатывало запросы.

In [ ]:
if errors.empty:
    print('Ошибки в логах не обнаружены.')
else:
    display(errors[['datetime', 'hypothesisId', 'message', 'error']])

## 8. Итоговый вывод

Сформируем короткое текстовое заключение для отчета.

In [ ]:
total_visits = len(visits)
max_count = int(visits['count'].max()) if not visits.empty else 0
error_count = len(errors)

top_decision = 'нет данных'
if not decision_stats.empty:
    top_decision = decision_stats.iloc[0]['Решение']

print('Итог анализа:')
print(f'- Зафиксировано посещений сайта: {total_visits}')
print(f'- Текущее значение счетчика Redis: {max_count}')
print(f'- Самый частый сценарий приложения: {top_decision}')
print(f'- Количество ошибок в логах: {error_count}')

if error_count == 0:
    print('- Приложение работает стабильно: критические ошибки не обнаружены.')
else:
    print('- В логах есть ошибки, требуется проверить подключение к Redis или состояние контейнеров.')